In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('/content/laptop_prices.csv', encoding='latin-1')
print("First 5 rows of the DataFrame:")
print(df.head())
print("\nConcise summary of the DataFrame:")
df.info()
print("Missing values before handling:")
print(df.isnull().sum())



# Distribution of Laptop Prices


In [ ]:

plt.figure(figsize=(10, 6))
sns.histplot(df['Price_euros'], kde=True, bins=30)
plt.title('Distribution of Laptop Prices (Price euros)')
plt.xlabel('Price (Euros)')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

categorical_cols = df.select_dtypes(include='object').columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print("First 5 rows of the encoded DataFrame:")
print(df_encoded.head())

from sklearn.feature_selection import SelectFromModel
X = df_encoded.drop('Price_euros', axis=1)
y = df_encoded['Price_euros']
estimator = RandomForestRegressor(random_state=42)
selector = SelectFromModel(estimator, threshold='median')
selector.fit(X, y)
X_selected = pd.DataFrame(selector.transform(X), columns=X.columns[(selector.get_support())])
print(f"Number of features before selection: {X.shape[1]}")
print(f"Number of features after selection: {X_selected.shape[1]}")

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, shuffle=True)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")
initial_model = RandomForestRegressor(random_state=42)
initial_model.fit(X_train, y_train)

y_pred = initial_model.predict(X_test)
y_pred_series = pd.Series(y_pred, index=y_test.index)
residuals_initial = y_test - y_pred_series
print("First 5 predicted values from y_pred_series:")
print(y_pred_series.head())
print("\nFirst 5 residuals from residuals_initial:")
print(residuals_initial.head())

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_test, y=y_pred_series, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Initial Model: Actual vs. Predicted Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)

plt.subplot(1, 2, 2)
sns.scatterplot(x=y_test, y=y_pred_series, alpha=0.6, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Tuned Model: Actual vs. Predicted Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)

plt.tight_layout()
plt.show()

y_pred_tuned_series = y_pred_series
residuals_tuned = y_test - y_pred_tuned_series

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_pred_series, y=residuals_initial, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.title('Initial Model: Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals (Actual - Predicted)')
plt.grid(True)

plt.subplot(1, 2, 2)
sns.scatterplot(x=y_pred_tuned_series, y=residuals_tuned, alpha=0.6, color='green')
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.title('Tuned Model: Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals (Actual - Predicted)')
plt.grid(True)
plt.tight_layout()
plt.show()

residuals_df = pd.DataFrame({'Residuals': pd.concat([residuals_initial, residuals_tuned]),'Model': ['Initial Model'] * len(residuals_initial) + ['Tuned Model'] * len(residuals_tuned)})

print(residuals_df.head())

plt.figure(figsize=(10, 6))
sns.boxplot(x='Model', y='Residuals', hue='Model', data=residuals_df, palette={'Initial Model': 'skyblue', 'Tuned Model': 'lightcoral'}, legend=False)
plt.title('Distribution of Residuals by Model')
plt.xlabel('Model')
plt.ylabel('Residuals (Actual - Predicted)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

importance_df = pd.DataFrame({'Feature': X_selected.columns,'Importance': initial_model.feature_importances_}).sort_values(by='Importance', ascending=False)
plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=importance_df.head(10), palette='viridis', legend=False)
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

mae_initial = mean_absolute_error(y_test, y_pred_series)
mse_initial = mean_squared_error(y_test, y_pred_series)
r2_initial = r2_score(y_test, y_pred_series)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned_series)
mse_tuned = mean_squared_error(y_test, y_pred_tuned_series)
r2_tuned = r2_score(y_test, y_pred_tuned_series)
metrics_df = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'R2'],
    'Initial Model': [mae_initial, mse_initial, r2_initial],
    'Tuned Model':   [mae_tuned,   mse_tuned,   r2_tuned]})

metrics_df_melted = metrics_df.melt(id_vars='Metric', var_name='Model', value_name='Value')
plt.figure(figsize=(10, 6))
sns.barplot(x='Metric', y='Value', hue='Model',data=metrics_df_melted, palette={'Initial Model': 'skyblue', 'Tuned Model': 'lightcoral'})
plt.title('Comparison of Evaluation Metrics: Initial vs. Tuned Model')
plt.xlabel('Metric')
plt.ylabel('Value')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

top_numerical_features = ['Ram', 'Weight', 'CPU_freq', 'PrimaryStorage']
plt.figure(figsize=(16, 12))
for i, feature in enumerate(top_numerical_features):
    plt.subplot(2, 2, i + 1)
    sns.scatterplot(x=df[feature], y=df['Price_euros'], alpha=0.6)
    plt.title(f'Laptop Price vs. {feature}')
    plt.xlabel(feature)
    plt.ylabel('Price (Euros)')
    plt.grid(True)

plt.tight_layout()
plt.show()

df['Company'].value_counts().plot(kind='bar', figsize=(12, 5))
plt.title('Laptop Distribution by Company')
plt.xlabel('Company')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df['OS'].value_counts().plot(kind='bar', figsize=(12, 5))
plt.title('Operating System Distribution')
plt.xlabel('Operating System')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df['Touchscreen'].value_counts().plot(kind='pie', autopct='%.2f%%', figsize=(8, 6))
plt.title('Touchscreen Availability')
plt.ylabel('')
plt.show()

df['Ram'].value_counts().sort_index().plot(kind='bar', figsize=(12, 5))
plt.title('RAM Distribution')
plt.xlabel('RAM (GB)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

df['CPU_company'].value_counts().plot(kind='pie', autopct='%.2f%%', figsize=(8, 6))
plt.title('CPU Company Distribution')
plt.ylabel('')
plt.show()

df['GPU_company'].value_counts().plot(kind='pie', autopct='%.2f%%', figsize=(8, 6))
plt.title('GPU Company Distribution')
plt.ylabel('')
plt.show()

df['PrimaryStorageType'].value_counts().plot(kind='pie', autopct='%.2f%%', figsize=(8, 6))
plt.title('Primary Storage Type Distribution')
plt.ylabel('')
plt.show()

df['Screen'].value_counts().plot(kind='bar', figsize=(10, 5))
plt.title('Screen Resolution Distribution')
plt.xlabel('Screen Definition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

df['SecondaryStorageType'].value_counts().plot(kind='bar', figsize=(10, 5))
plt.title('Secondary Storage Type Distribution')
plt.xlabel('Storage Type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
sns.boxplot(x='OS', y='Price_euros', data=df)
plt.title('Operating System vs Laptop Price')
plt.xlabel('Operating System')
plt.ylabel('Price (Euros)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(x='Touchscreen', y='Price_euros', hue='Screen', data=df)
plt.title('Touchscreen vs Price by Screen Type')
plt.xlabel('Touchscreen')
plt.ylabel('Price (Euros)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(x='OS', y='Price_euros', hue='Touchscreen', data=df)
plt.title('OS vs Price by Touchscreen')
plt.xlabel('Operating System')
plt.ylabel('Price (Euros)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(x='PrimaryStorage', y='Price_euros', hue='SecondaryStorage', data=df)
plt.title('Primary Storage vs Price by Secondary Storage')
plt.xlabel('Primary Storage (GB)')
plt.ylabel('Price (Euros)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(x='RetinaDisplay', y='Price_euros', hue='PrimaryStorage', data=df)
plt.title('Retina Display vs Price by Primary Storage')
plt.xlabel('Retina Display')
plt.ylabel('Price (Euros)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 7))
sns.scatterplot(data=df, x='CPU_freq', y='Ram', hue='OS', s=100)
plt.title('CPU Frequency vs RAM by Operating System')
plt.xlabel('CPU Frequency (GHz)')
plt.ylabel('RAM (GB)')
plt.tight_layout()
plt.show()

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
correlation_matrix = df[numerical_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

df['IPSpanel'].value_counts().plot(kind='pie', autopct='%.1f%%', figsize=(8, 6))
plt.title('IPS Panel Availability')
plt.ylabel('')
plt.show()

df['Inches'].value_counts().sort_index().plot(kind='bar', figsize=(14, 5))
plt.title('Screen Size Distribution')
plt.xlabel('Screen Size (Inches)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
